# Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import string

import nltk
import spacy

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Download NLTK Resources


In [2]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

# Upload Datasets

In [3]:
from google.colab import files

upload= files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv
Saving bbc_text_cls.csv to bbc_text_cls.csv


# Read BBC Dataset

In [4]:
bbc=pd.read_csv("bbc_text_cls.csv")

bbc.head()

,text,labels
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business


# Prepare BBC Dataset

In [5]:
X=bbc["text"].fillna("").astype(str)

y=bbc["labels"]

X_train_bbc, X_test_bbc, y_train_bbc, y_test_bbc = train_test_split( X,  y, test_size=0.2, random_state=42, stratify=y)
X_train_bbc.head()

print("Training Data :", len(X_train_bbc))
print("Testing Data :", len(X_test_bbc))

Training Data : 1780
Testing Data : 445


# Case Folding

In [6]:
X_train_bbc=X_train_bbc.str.lower()
X_test_bbc=X_test_bbc.str.lower()
X_train_bbc.head()

,text
141,asian banks halt dollar's slide\n\nthe dollar ...
1399,gb quartet get cross country call\n\nfour brit...
807,spider-man creator wins profits\n\nspider-man ...
1054,howard unveils election platform\n\nthe conser...
1080,police probe bnp mosque leaflet\n\npolice are ...


# Tokenization

In [7]:
X_train_bbc=X_train_bbc.apply(lambda x: word_tokenize(str(x)))
X_test_bbc=X_test_bbc.apply(lambda x: word_tokenize(str(x)))

X_train_bbc.head()

,text
141,"[asian, banks, halt, dollar, 's, slide, the, d..."
1399,"[gb, quartet, get, cross, country, call, four,..."
807,"[spider-man, creator, wins, profits, spider-ma..."
1054,"[howard, unveils, election, platform, the, con..."
1080,"[police, probe, bnp, mosque, leaflet, police, ..."


# Synonym Substitution

In [8]:
def get_synonym(word):


    synsets = wordnet.synsets(word)


    for synset in synsets:


        for lemma in synset.lemmas():


            synonym = lemma.name().replace("_", " ").lower()


            if synonym != word.lower():


                return synonym


    return word


X_train_bbc = X_train_bbc.apply(
    lambda words: [get_synonym(word) for word in words]
)


X_test_bbc = X_test_bbc.apply(
    lambda words: [get_synonym(word) for word in words]
)


X_train_bbc.head()


,text
141,"[asiatic, sir joseph banks, arrest, dollar bil..."
1399,"[sarin, four, acquire, crisscross, state, phon..."
807,"[spider-man, godhead, win, net income, spider-..."
1054,"[leslie howard, unveil, election, political pl..."
1080,"[police force, investigation, bnp, mosque, cus..."


# Punctuation Removal

In [10]:
X_train_bbc=X_train_bbc.apply(lambda words: [word for word in words if word not in string.punctuation])

X_test_bbc=X_test_bbc.apply(lambda words: [word for word in words if word not in string.punctuation])

X_train_bbc.head()

,text
141,"[asiatic, sir joseph banks, arrest, dollar bil..."
1399,"[sarin, four, acquire, crisscross, state, phon..."
807,"[spider-man, godhead, win, net income, spider-..."
1054,"[leslie howard, unveil, election, political pl..."
1080,"[police force, investigation, bnp, mosque, cus..."


# Stop Words Removal

In [11]:
stop_words = set(stopwords.words("english"))

X_train_bbc = X_train_bbc.apply(
    lambda words: [word for word in words if word not in stop_words]
)

X_test_bbc = X_test_bbc.apply(
    lambda words: [word for word in words if word not in stop_words]
)

X_train_bbc.head()

stop_words=set(stopwords.words("english"))

X_train_bbc=X_train_bbc.apply(lambda words: [word for word in words if word not in stop_words])
X_test_bbc=X_test_bbc.apply(lambda words: [word for word in words if word not in stop_words])

X_train_bbc.head()

,text
141,"[asiatic, sir joseph banks, arrest, dollar bil..."
1399,"[sarin, four, acquire, crisscross, state, phon..."
807,"[spider-man, godhead, win, net income, spider-..."
1054,"[leslie howard, unveil, election, political pl..."
1080,"[police force, investigation, bnp, mosque, cus..."


# Stemming

In [15]:
stemmer=PorterStemmer()

X_train_bbc=X_train_bbc.apply(lambda words: [stemmer.stem(word) for word in words])
X_test_bbc=X_test_bbc.apply(lambda words: [stemmer.stem(word) for word in words])

X_train_bbc.head()

,text
141,"[asiat, sir joseph bank, arrest, dollar bil, '..."
1399,"[sarin, four, acquir, crisscross, state, phone..."
807,"[spider-man, godhead, win, net incom, spider-m..."
1054,"[leslie howard, unveil, elect, political platf..."
1080,"[police forc, investig, bnp, mosqu, cusp, poli..."


# Lemmatization

In [29]:
lemmatizer=WordNetLemmatizer()

X_train_bbc=X_train_bbc.apply(lambda words: [lemmatizer.lemmatize(word) for word in words])
X_test_bbc=X_test_bbc.apply(lambda words: [lemmatizer.lemmatize(word) for word in words])

X_train_bbc.head()

,text
141,"[asiat, sir joseph bank, arrest, dollar bil, '..."
1399,"[sarin, four, acquir, crisscross, state, phone..."
807,"[spider-man, godhead, win, net incom, spider-m..."
1054,"[leslie howard, unveil, elect, political platf..."
1080,"[police forc, investig, bnp, mosqu, cusp, poli..."


# Bag of Words

In [30]:
X_train_bbc = X_train_bbc.apply(lambda words: " ".join(words))
X_test_bbc = X_test_bbc.apply(lambda words: " ".join(words))

bbc_bow = CountVectorizer()

X_train_bbc_bow = bbc_bow.fit_transform(X_train_bbc)

X_test_bbc_bow = bbc_bow.transform(X_test_bbc)

print(X_train_bbc_bow.shape)
print(X_test_bbc_bow.shape)


(1780, 18953)
(445, 18953)


# Bow + MultinomialNB

In [32]:

bbc_nb=MultinomialNB()
bbc_nb.fit(X_train_bbc_bow, y_train_bbc)
y_pred_nb=bbc_nb.predict(X_test_bbc_bow)
bbc_nb_bow_accuracy=accuracy_score(y_test_bbc,y_pred_nb)
print("Accuracy :", bbc_nb_bow_accuracy)
print("Precision :", precision_score(y_test_bbc, y_pred_nb, average="weighted"))
print("Recall :", recall_score(y_test_bbc, y_pred_nb, average="weighted"))
print("F1 Score :", f1_score(y_test_bbc, y_pred_nb, average="weighted"))

Accuracy : 0.9797752808988764
Precision : 0.9803402501882343
Recall : 0.9797752808988764
F1 Score : 0.9798277523965595


# Bow + Logistic Regression

In [33]:

bbc_lr=LogisticRegression(max_iter=1000)

bbc_lr.fit(X_train_bbc_bow, y_train_bbc)
y_pred_lr=bbc_lr.predict(X_test_bbc_bow)
bbc_lr_bow_accuracy=accuracy_score(y_test_bbc, y_pred_lr)
print("Accuracy :", bbc_lr_bow_accuracy)
print("Precision :", precision_score(y_test_bbc, y_pred_lr, average="weighted"))
print("Recall :", recall_score(y_test_bbc, y_pred_lr, average="weighted"))
print("F1 Score :", f1_score(y_test_bbc, y_pred_lr, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_bbc, y_pred_lr))

print("\nClassification Report")
print(classification_report(y_test_bbc, y_pred_lr))

Accuracy : 0.9752808988764045
Precision : 0.9754375852536381
Recall : 0.9752808988764045
F1 Score : 0.9751330175180781

Confusion Matrix
[[ 99   1   1   0   1]
 [  0  77   0   0   0]
 [  1   1  78   3   1]
 [  1   0   0 101   0]
 [  1   0   0   0  79]]

Classification Report
               precision    recall  f1-score   support

     business       0.97      0.97      0.97       102
entertainment       0.97      1.00      0.99        77
     politics       0.99      0.93      0.96        84
        sport       0.97      0.99      0.98       102
         tech       0.98      0.99      0.98        80

     accuracy                           0.98       445
    macro avg       0.98      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445



# Bow - LinearSVC

In [35]:

bbc_svm=LinearSVC()

bbc_svm.fit(X_train_bbc_bow, y_train_bbc)
y_pred_svm=bbc_svm.predict(X_test_bbc_bow)
bbc_svm_accuracy=accuracy_score(y_test_bbc, y_pred_svm)
print("Accuracy :", bbc_svm_accuracy)
print("Precision :", precision_score(y_test_bbc, y_pred_svm, average="weighted"))
print("Recall :", recall_score(y_test_bbc, y_pred_svm, average="weighted"))
print("F1 Score :", f1_score(y_test_bbc, y_pred_svm, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_bbc, y_pred_svm))

print("\nClassification Report")
print(classification_report(y_test_bbc, y_pred_svm))

Accuracy : 0.9752808988764045
Precision : 0.9756216622425853
Recall : 0.9752808988764045
F1 Score : 0.9752008832645958

Confusion Matrix
[[ 98   1   1   1   1]
 [  0  77   0   0   0]
 [  1   2  79   1   1]
 [  0   1   0 101   0]
 [  1   0   0   0  79]]

Classification Report
               precision    recall  f1-score   support

     business       0.98      0.96      0.97       102
entertainment       0.95      1.00      0.97        77
     politics       0.99      0.94      0.96        84
        sport       0.98      0.99      0.99       102
         tech       0.98      0.99      0.98        80

     accuracy                           0.98       445
    macro avg       0.97      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445



/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


# TF-IDF

In [37]:
bbc_tfidf = TfidfVectorizer()

X_train_bbc_tfidf = bbc_tfidf.fit_transform(X_train_bbc)

X_test_bbc_tfidf = bbc_tfidf.transform(X_test_bbc)

print(X_train_bbc_tfidf.shape)
print(X_test_bbc_tfidf.shape)



(1780, 18953)
(445, 18953)


# TF-IDF + MultinomialNB

In [ ]:
bbc_nb = MultinomialNB()

bbc_nb.fit(X_train_bbc_tfidf, y_train_bbc)

y_pred_nb = bbc_nb.predict(X_test_bbc_tfidf)

bbc_nb_tfidf_accuracy = accuracy_score(y_test_bbc, y_pred_nb)


print("Accuracy :", bbc_nb_tfidf_accuracy)
print("Precision :", precision_score(y_test_bbc, y_pred_nb, average="weighted"))
print("Recall :", recall_score(y_test_bbc, y_pred_nb, average="weighted"))
print("F1 Score :", f1_score(y_test_bbc, y_pred_nb, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_bbc, y_pred_nb))

print("\nClassification Report")
print(classification_report(y_test_bbc, y_pred_nb))

Accuracy : 0.9707865168539326
Precision : 0.9715188848809024
Recall : 0.9707865168539326
F1 Score : 0.9707444733422405

Confusion Matrix
[[ 98   0   3   0   1]
 [  1  71   3   1   1]
 [  1   0  82   0   1]
 [  0   0   0 102   0]
 [  1   0   0   0  79]]

Classification Report
               precision    recall  f1-score   support

     business       0.97      0.96      0.97       102
entertainment       1.00      0.92      0.96        77
     politics       0.93      0.98      0.95        84
        sport       0.99      1.00      1.00       102
         tech       0.96      0.99      0.98        80

     accuracy                           0.97       445
    macro avg       0.97      0.97      0.97       445
 weighted avg       0.97      0.97      0.97       445



# TF-IDF + Logistic Regression

In [ ]:
bbc_lr = LogisticRegression(max_iter=1000)

bbc_lr.fit(X_train_bbc_tfidf, y_train_bbc)

y_pred_lr = bbc_lr.predict(X_test_bbc_tfidf)

bbc_lr_tfidf_accuracy = accuracy_score(y_test_bbc, y_pred_lr)

print("Accuracy :", bbc_lr_tfidf_accuracy)
print("Precision :", precision_score(y_test_bbc, y_pred_lr, average="weighted"))
print("Recall :", recall_score(y_test_bbc, y_pred_lr, average="weighted"))
print("F1 Score :", f1_score(y_test_bbc, y_pred_lr, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_bbc, y_pred_lr))

print("\nClassification Report")
print(classification_report(y_test_bbc, y_pred_lr))

Accuracy : 0.9820224719101124
Precision : 0.9820768077733
Recall : 0.9820224719101124
F1 Score : 0.9819440870612228

Confusion Matrix
[[ 99   0   1   1   1]
 [  0  77   0   0   0]
 [  2   0  80   1   1]
 [  0   1   0 101   0]
 [  0   0   0   0  80]]

Classification Report
               precision    recall  f1-score   support

     business       0.98      0.97      0.98       102
entertainment       0.99      1.00      0.99        77
     politics       0.99      0.95      0.97        84
        sport       0.98      0.99      0.99       102
         tech       0.98      1.00      0.99        80

     accuracy                           0.98       445
    macro avg       0.98      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445



# TF-IDF + LinearSVC

In [ ]:
bbc_svm = LinearSVC()

bbc_svm.fit(X_train_bbc_tfidf, y_train_bbc)

y_pred_svm = bbc_svm.predict(X_test_bbc_tfidf)

bbc_svm_tfidf_accuracy = accuracy_score(y_test_bbc, y_pred_svm)

print("Accuracy :", bbc_svm_tfidf_accuracy)
print("Precision :", precision_score(y_test_bbc, y_pred_svm, average="weighted"))
print("Recall :", recall_score(y_test_bbc, y_pred_svm, average="weighted"))
print("F1 Score :", f1_score(y_test_bbc, y_pred_svm, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_bbc, y_pred_svm))

print("\nClassification Report")
print(classification_report(y_test_bbc, y_pred_svm))

Accuracy : 0.9842696629213483
Precision : 0.9845096240048413
Recall : 0.9842696629213483
F1 Score : 0.9842325870316302

Confusion Matrix
[[ 98   1   1   1   1]
 [  0  77   0   0   0]
 [  0   0  82   1   1]
 [  0   1   0 101   0]
 [  0   0   0   0  80]]

Classification Report
               precision    recall  f1-score   support

     business       1.00      0.96      0.98       102
entertainment       0.97      1.00      0.99        77
     politics       0.99      0.98      0.98        84
        sport       0.98      0.99      0.99       102
         tech       0.98      1.00      0.99        80

     accuracy                           0.98       445
    macro avg       0.98      0.99      0.98       445
 weighted avg       0.98      0.98      0.98       445



# Read AG News Dataset

In [ ]:
ag_train = pd.read_csv("train.csv", header=None)

ag_test = pd.read_csv("test.csv", header=None)

ag_train.columns = ["label", "title", "description"]

ag_test.columns = ["label", "title", "description"]

ag_train["text"] = ag_train["title"].fillna("") + " " + ag_train["description"].fillna("")

ag_test["text"] = ag_test["title"].fillna("") + " " + ag_test["description"].fillna("")

X_train_ag = ag_train["text"]

y_train_ag = ag_train["label"]

X_test_ag = ag_test["text"]

y_test_ag = ag_test["label"]

ag_train.head()

,label,title,description,text
0,Class Index,Title,Description,Title Description
1,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...
2,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...
3,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...
4,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...


# Case Folding

In [ ]:
X_train_ag = X_train_ag.str.lower()

X_test_ag = X_test_ag.str.lower()

X_train_ag.head()

,text
0,title description
1,wall st. bears claw back into the black (reute...
2,carlyle looks toward commercial aerospace (reu...
3,oil and economy cloud stocks' outlook (reuters...
4,iraq halts oil exports from main southern pipe...


# Tokenization

In [ ]:
X_train_ag = X_train_ag.apply(word_tokenize)

X_test_ag = X_test_ag.apply(word_tokenize)

X_train_ag.head()

,text
0,"[title, description]"
1,"[wall, st., bears, claw, back, into, the, blac..."
2,"[carlyle, looks, toward, commercial, aerospace..."
3,"[oil, and, economy, cloud, stocks, ', outlook,..."
4,"[iraq, halts, oil, exports, from, main, southe..."


# Synonym Substitution

In [ ]:
X_train_ag = X_train_ag.apply(
    lambda words: [get_synonym(word) for word in words]
)


X_test_ag = X_test_ag.apply(
    lambda words: [get_synonym(word) for word in words]
)


X_train_ag.head()


,text
0,"[statute title, verbal description]"
1,"[paries, st., bear, hook, dorsum, into, the, b..."
2,"[thomas carlyle, expression, toward, commercia..."
3,"[oil color, and, economic system, swarm, stock..."
4,"[republic of iraq, arrest, oil color, export, ..."


# Punctuation Removal

In [ ]:
X_train_ag = X_train_ag.apply(
    lambda words: [word for word in words if word not in string.punctuation]
)

X_test_ag = X_test_ag.apply(
    lambda words: [word for word in words if word not in string.punctuation]
)

X_train_ag.head()

,text
0,"[statute title, verbal description]"
1,"[paries, st., bear, hook, dorsum, into, the, b..."
2,"[thomas carlyle, expression, toward, commercia..."
3,"[oil color, and, economic system, swarm, stock..."
4,"[republic of iraq, arrest, oil color, export, ..."


# Stop Words Removal

In [ ]:
stop_words = set(stopwords.words("english"))

X_train_ag = X_train_ag.apply(
    lambda words: [word for word in words if word not in stop_words]
)

X_test_ag = X_test_ag.apply(
    lambda words: [word for word in words if word not in stop_words]
)

X_train_ag.head()

,text
0,"[statute title, verbal description]"
1,"[paries, st., bear, hook, dorsum, blackness, r..."
2,"[thomas carlyle, expression, toward, commercia..."
3,"[oil color, economic system, swarm, stock, men..."
4,"[republic of iraq, arrest, oil color, export, ..."


# Stemming

In [ ]:
stemmer = PorterStemmer()

X_train_ag = X_train_ag.apply(
    lambda words: [stemmer.stem(word) for word in words]
)

X_test_ag = X_test_ag.apply(
    lambda words: [stemmer.stem(word) for word in words]
)

X_train_ag.head()

,text
0,"[statute titl, verbal descript]"
1,"[pari, st., bear, hook, dorsum, black, reuter,..."
2,"[thomas carlyl, express, toward, commercial me..."
3,"[oil color, economic system, swarm, stock, men..."
4,"[republic of iraq, arrest, oil color, export, ..."


# Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()

X_train_ag = X_train_ag.apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words]
)

X_test_ag = X_test_ag.apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words]
)

X_train_ag = X_train_ag.apply(lambda words: " ".join(words))

X_test_ag = X_test_ag.apply(lambda words: " ".join(words))

X_train_ag.head()

,text
0,statute titl verbal descript
1,pari st. bear hook dorsum black reuter reuter ...
2,thomas carlyl express toward commercial messag...
3,oil color economic system swarm stock mental r...
4,republic of iraq arrest oil color export brini...


# Bag of Words

In [ ]:
ag_bow = CountVectorizer()

X_train_ag_bow = ag_bow.fit_transform(X_train_ag)

X_test_ag_bow = ag_bow.transform(X_test_ag)

print(X_train_ag_bow.shape)

print(X_test_ag_bow.shape)

(120001, 48664)
(7601, 48664)


# Bag of Words + MultinomialNB

In [ ]:
ag_nb = MultinomialNB()

ag_nb.fit(X_train_ag_bow, y_train_ag)

y_pred_nb = ag_nb.predict(X_test_ag_bow)

ag_nb_bow_accuracy = accuracy_score(y_test_ag, y_pred_nb)

print("Accuracy :", ag_nb_bow_accuracy)
print("Precision :", precision_score(y_test_ag, y_pred_nb, average="weighted"))
print("Recall :", recall_score(y_test_ag, y_pred_nb, average="weighted"))
print("F1 Score :", f1_score(y_test_ag, y_pred_nb, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_ag, y_pred_nb))

print("\nClassification Report")
print(classification_report(y_test_ag, y_pred_nb))

Accuracy : 0.892777266149191
Precision : 0.8921775065108108
Recall : 0.892777266149191
F1 Score : 0.892307047826199

Confusion Matrix
[[1697   63   88   52    0]
 [  27 1849   11   13    0]
 [  87   24 1579  210    0]
 [  72   22  145 1661    0]
 [   0    1    0    0    0]]

Classification Report
              precision    recall  f1-score   support

           1       0.90      0.89      0.90      1900
           2       0.94      0.97      0.96      1900
           3       0.87      0.83      0.85      1900
           4       0.86      0.87      0.87      1900
 Class Index       0.00      0.00      0.00         1

    accuracy                           0.89      7601
   macro avg       0.71      0.71      0.71      7601
weighted avg       0.89      0.89      0.89      7601



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

# Bag of Words + Logistic Regression

In [ ]:
ag_lr = LogisticRegression(max_iter=1000)

ag_lr.fit(X_train_ag_bow, y_train_ag)

y_pred_lr = ag_lr.predict(X_test_ag_bow)

ag_lr_bow_accuracy = accuracy_score(y_test_ag, y_pred_lr)

print("Accuracy :", ag_lr_bow_accuracy)
print("Precision :", precision_score(y_test_ag, y_pred_lr, average="weighted"))
print("Recall :", recall_score(y_test_ag, y_pred_lr, average="weighted"))
print("F1 Score :", f1_score(y_test_ag, y_pred_lr, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_ag, y_pred_lr))

print("\nClassification Report")
print(classification_report(y_test_ag, y_pred_lr))

Accuracy : 0.9013287725299303
Precision : 0.9010920038825603
Recall : 0.9013287725299303
F1 Score : 0.9011379701935255

Confusion Matrix
[[1696   55   83   66    0]
 [  30 1848   14    8    0]
 [  69   18 1648  165    0]
 [  54   21  166 1659    0]
 [   0    0    0    1    0]]

Classification Report
              precision    recall  f1-score   support

           1       0.92      0.89      0.90      1900
           2       0.95      0.97      0.96      1900
           3       0.86      0.87      0.86      1900
           4       0.87      0.87      0.87      1900
 Class Index       0.00      0.00      0.00         1

    accuracy                           0.90      7601
   macro avg       0.72      0.72      0.72      7601
weighted avg       0.90      0.90      0.90      7601



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

# Bag of Words + LinearSVC

In [ ]:
ag_svm = LinearSVC()

ag_svm.fit(X_train_ag_bow, y_train_ag)

y_pred_svm = ag_svm.predict(X_test_ag_bow)

ag_svm_bow_accuracy = accuracy_score(y_test_ag, y_pred_svm)

print("Accuracy :", ag_svm_bow_accuracy)
print("Precision :", precision_score(y_test_ag, y_pred_svm, average="weighted"))
print("Recall :", recall_score(y_test_ag, y_pred_svm, average="weighted"))
print("F1 Score :", f1_score(y_test_ag, y_pred_svm, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_ag, y_pred_svm))

print("\nClassification Report")
print(classification_report(y_test_ag, y_pred_svm))

Accuracy : 0.8922510196026838
Precision : 0.8922431456684327
Recall : 0.8922510196026838
F1 Score : 0.8922163500838456

Confusion Matrix
[[1688   54   92   66    0]
 [  50 1819   16   15    0]
 [  73   22 1620  185    0]
 [  54   23  169 1654    0]
 [   0    0    0    0    1]]

Classification Report
              precision    recall  f1-score   support

           1       0.91      0.89      0.90      1900
           2       0.95      0.96      0.95      1900
           3       0.85      0.85      0.85      1900
           4       0.86      0.87      0.87      1900
 Class Index       1.00      1.00      1.00         1

    accuracy                           0.89      7601
   macro avg       0.91      0.91      0.91      7601
weighted avg       0.89      0.89      0.89      7601



# TF-IDF

In [ ]:
ag_tfidf = TfidfVectorizer()

X_train_ag_tfidf = ag_tfidf.fit_transform(X_train_ag)

X_test_ag_tfidf = ag_tfidf.transform(X_test_ag)

print(X_train_ag_tfidf.shape)

print(X_test_ag_tfidf.shape)

(120001, 48664)
(7601, 48664)


# TF-IDF + MultinomialNB

In [ ]:
ag_nb = MultinomialNB()

ag_nb.fit(X_train_ag_tfidf, y_train_ag)

y_pred_nb = ag_nb.predict(X_test_ag_tfidf)

ag_nb_tfidf_accuracy = accuracy_score(y_test_ag, y_pred_nb)

print("Accuracy :", ag_nb_tfidf_accuracy)
print("Precision :", precision_score(y_test_ag, y_pred_nb, average="weighted"))
print("Recall :", recall_score(y_test_ag, y_pred_nb, average="weighted"))
print("F1 Score :", f1_score(y_test_ag, y_pred_nb, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_ag, y_pred_nb))

print("\nClassification Report")
print(classification_report(y_test_ag, y_pred_nb))

Accuracy : 0.8986975397973951
Precision : 0.8980795134803389
Recall : 0.8986975397973951
F1 Score : 0.8983059340655878

Confusion Matrix
[[1699   61   94   46    0]
 [  26 1853   11   10    0]
 [  79   23 1620  178    0]
 [  73   23  145 1659    0]
 [   0    1    0    0    0]]

Classification Report
              precision    recall  f1-score   support

           1       0.91      0.89      0.90      1900
           2       0.94      0.98      0.96      1900
           3       0.87      0.85      0.86      1900
           4       0.88      0.87      0.87      1900
 Class Index       0.00      0.00      0.00         1

    accuracy                           0.90      7601
   macro avg       0.72      0.72      0.72      7601
weighted avg       0.90      0.90      0.90      7601



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

# TF-IDF + Logistic Regression

In [ ]:
ag_lr = LogisticRegression(max_iter=1000)

ag_lr.fit(X_train_ag_tfidf, y_train_ag)

y_pred_lr = ag_lr.predict(X_test_ag_tfidf)

ag_lr_tfidf_accuracy = accuracy_score(y_test_ag, y_pred_lr)

print("Accuracy :", ag_lr_tfidf_accuracy)
print("Precision :", precision_score(y_test_ag, y_pred_lr, average="weighted"))
print("Recall :", recall_score(y_test_ag, y_pred_lr, average="weighted"))
print("F1 Score :", f1_score(y_test_ag, y_pred_lr, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_ag, y_pred_lr))

print("\nClassification Report")
print(classification_report(y_test_ag, y_pred_lr))

Accuracy : 0.9121168267333246
Precision : 0.9117829504292727
Recall : 0.9121168267333246
F1 Score : 0.9118462136021

Confusion Matrix
[[1717   54   77   52    0]
 [  16 1862   14    8    0]
 [  61   21 1665  153    0]
 [  54   22  135 1689    0]
 [   0    1    0    0    0]]

Classification Report
              precision    recall  f1-score   support

           1       0.93      0.90      0.92      1900
           2       0.95      0.98      0.96      1900
           3       0.88      0.88      0.88      1900
           4       0.89      0.89      0.89      1900
 Class Index       0.00      0.00      0.00         1

    accuracy                           0.91      7601
   macro avg       0.73      0.73      0.73      7601
weighted avg       0.91      0.91      0.91      7601



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

# TF-IDF + LinearSVC

In [ ]:
ag_svm = LinearSVC()

ag_svm.fit(X_train_ag_tfidf, y_train_ag)

y_pred_svm = ag_svm.predict(X_test_ag_tfidf)

ag_svm_tfidf_accuracy = accuracy_score(y_test_ag, y_pred_svm)

print("Accuracy :", ag_svm_tfidf_accuracy)
print("Precision :", precision_score(y_test_ag, y_pred_svm, average="weighted"))
print("Recall :", recall_score(y_test_ag, y_pred_svm, average="weighted"))
print("F1 Score :", f1_score(y_test_ag, y_pred_svm, average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_ag, y_pred_svm))

print("\nClassification Report")
print(classification_report(y_test_ag, y_pred_svm))

Accuracy : 0.9172477305617682
Precision : 0.9172537275412069
Recall : 0.9172477305617682
F1 Score : 0.9171295139314662

Confusion Matrix
[[1719   52   78   51    0]
 [  17 1863   12    8    0]
 [  51   19 1676  154    0]
 [  46   13  128 1713    0]
 [   0    0    0    0    1]]

Classification Report
              precision    recall  f1-score   support

           1       0.94      0.90      0.92      1900
           2       0.96      0.98      0.97      1900
           3       0.88      0.88      0.88      1900
           4       0.89      0.90      0.90      1900
 Class Index       1.00      1.00      1.00         1

    accuracy                           0.92      7601
   macro avg       0.93      0.93      0.93      7601
weighted avg       0.92      0.92      0.92      7601



# Fianl result table

In [ ]:
results = pd.DataFrame({

    "Algorithm": [
        "Naive Bayes",
        "Naive Bayes",
        "Logistic Regression",
        "Logistic Regression",
        "Support Vector Machine",
        "Support Vector Machine"
    ],

    "Representation": [
        "BoW",
        "TF-IDF",
        "BoW",
        "TF-IDF",
        "BoW",
        "TF-IDF"
    ],

    "Dataset 1": [
        round(bbc_nb_bow_accuracy, 4),
        round(bbc_nb_tfidf_accuracy, 4),
        round(bbc_lr_bow_accuracy, 4),
        round(bbc_lr_tfidf_accuracy, 4),
        round(bbc_svm_bow_accuracy, 4),
        round(bbc_svm_tfidf_accuracy, 4)
    ],

    "Dataset 2": [
        round(ag_nb_bow_accuracy, 4),
        round(ag_nb_tfidf_accuracy, 4),
        round(ag_lr_bow_accuracy, 4),
        round(ag_lr_tfidf_accuracy, 4),
        round(ag_svm_bow_accuracy, 4),
        round(ag_svm_tfidf_accuracy, 4)
    ]

})

results

,Algorithm,Representation,Dataset 1,Dataset 2
0,Naive Bayes,BoW,0.9843,0.8928
1,Naive Bayes,TF-IDF,0.9708,0.8987
2,Logistic Regression,BoW,0.9775,0.9013
3,Logistic Regression,TF-IDF,0.9820,0.9121
4,Support Vector Machine,BoW,0.9820,0.8923
5,Support Vector Machine,TF-IDF,0.9843,0.9172


# Testing

In [ ]:
from google.colab import files

uploaded = files.upload()

filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as f:
    new_document = f.read()

Saving entertainment_test.txt to entertainment_test.txt


# Testing result

In [ ]:
new_document = new_document.lower()

new_document = word_tokenize(new_document)

new_document = [get_synonym(word) for word in new_document]

new_document = [word for word in new_document if word not in string.punctuation]

new_document = [word for word in new_document if word not in stop_words]

new_document = [stemmer.stem(word) for word in new_document]

new_document = [lemmatizer.lemmatize(word) for word in new_document]

new_document = " ".join(new_document)

# done


In [ ]:
print(results)

                Algorithm Representation  Dataset 1  Dataset 2
0             Naive Bayes            BoW     0.9843     0.8928
1             Naive Bayes         TF-IDF     0.9708     0.8987
2     Logistic Regression            BoW     0.9775     0.9013
3     Logistic Regression         TF-IDF     0.9820     0.9121
4  Support Vector Machine            BoW     0.9820     0.8923
5  Support Vector Machine         TF-IDF     0.9843     0.9172


# New Section

In [ ]:
vector = bbc_tfidf.transform([new_document])

prediction = bbc_svm.predict(vector)

print("Predicted Document Type:", prediction[0])

Predicted Document Type: entertainment
